# Phase E: コロナ前後5年比較（2018〜2022）

- 2022年ファイル: 2020・2021・2022年価格
- 2020年ファイル: 2018・2019年価格
- `serial_id` で結合 → 5年分パネルデータを構築
- コロナ前（2018→2019）をベースラインに、コロナ禍の変動を比較

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    import japanize_matplotlib
except ImportError:
    pass

from compare_years import (
    build_panel, yearly_summary, by_district_yearly,
    by_prefecture_yearly, donut_effect_panel
)

plt.rcParams["figure.dpi"] = 120

# ── パス設定（ここだけ変えれば実データに切り替え可能）──────────────────
PATH_2022 = Path("/Users/KASU/REX_data_2020-2022/2022/nouhin_line_2022.shp")
PATH_2020 = Path("/Users/KASU/REX_data_2020-2022/2020/nouhin_line_2020.shp")
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

## 1. データ読み込み・5年パネル構築

In [ ]:
panel = build_panel(PATH_2022, PATH_2020)
print(f"\n件数: {len(panel):,}")
print(f"CRS : {panel.crs}")

## 2. 全国年次変化率サマリー

コロナ前（2018→2019）をベースラインとして、各年の変化率を比較する。

In [ ]:
summary = yearly_summary(panel)
print(summary.to_string())

In [ ]:
# 年次変化率の推移グラフ
years  = ["2018→2019", "2019→2020", "2020→2021", "2021→2022"]
cols   = ["chg_18_19", "chg_19_20", "chg_20_21", "chg_21_22"]
medians = [panel[c].median() for c in cols]
means   = [panel[c].mean()   for c in cols]

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(years))
ax.bar(x, medians, color=["#4A90D9" if v >= 0 else "#E05C5C" for v in medians],
       alpha=0.85, label="中央値")
ax.plot(x, means, marker="o", color="#1A3A5C", linewidth=2, label="平均", zorder=5)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

# コロナ禍の背景色
ax.axvspan(0.5, 3.5, alpha=0.06, color="orange", label="コロナ禍（2020〜2022）")
ax.set_xticks(x)
ax.set_xticklabels(years, fontsize=10)
ax.set_ylabel("路線価変化率 (%)")
ax.set_title("全国路線価 年次変化率の推移（2018〜2022）")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.2f}%"))
plt.tight_layout()
plt.savefig(OUT_DIR / "corona_yearly_change.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {OUT_DIR / 'corona_yearly_change.png'}")

### 2-2. 上昇・下落・据え置き 年次構成比

中央値が0になる原因として「据え置き路線の多さ」を確認する。

## 3. 地区区分別 × 年次比較

商業系・住宅系・工業系でコロナの影響がどう異なるか。

In [ ]:
dist_table = by_district_yearly(panel)
print(dist_table.to_string())

In [ ]:
# 地区別ヒートマップ
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(10, 3.5))
data = dist_table.values.astype(float)
abs_max = np.nanpercentile(np.abs(data), 95)
im = ax.imshow(data, cmap="RdBu", aspect="auto",
               vmin=-abs_max, vmax=abs_max)

ax.set_xticks(range(len(dist_table.columns)))
ax.set_xticklabels([c.replace("\n", " ") for c in dist_table.columns], fontsize=9)
ax.set_yticks(range(len(dist_table.index)))
ax.set_yticklabels(dist_table.index, fontsize=10)

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:+.2f}%", ha="center", va="center",
                    fontsize=9, color="white" if abs(val) > abs_max * 0.5 else "black")

fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04, label="変化率中央値 (%)")
ax.set_title("地区区分別 路線価変化率（中央値）ヒートマップ", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "corona_district_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {OUT_DIR / 'corona_district_heatmap.png'}")

## 4. Donut Effect 検証（コロナ前後比較）

都市中心からの距離帯別に変化率を集計し、
コロナ前（2018→2019）とコロナ禍（2019→2022）を比較する。

- `donut_diff` が負 → コロナ禍にコロナ前より下落した（都心に多いはず）
- `donut_diff` が正 → コロナ禍にコロナ前より上昇した（郊外に多いはず）

In [ ]:
donut = donut_effect_panel(panel, city="東京")
print(donut.to_string())

In [ ]:
# Donut Effect グラフ
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bands  = donut.index.astype(str)
x      = range(len(bands))

# 左: 距離帯別・年次変化率の推移
ax = axes[0]
cols_plot = ["pre_median", "p1920_median", "p2021_median", "p2122_median"]
labels_p  = ["2018→2019（前）", "2019→2020（発生）", "2020→2021（禍）", "2021→2022（回復）"]
colors_p  = ["#888888", "#E05C5C", "#C0392B", "#4A90D9"]
for col, label, color in zip(cols_plot, labels_p, colors_p):
    ax.plot(x, donut[col], marker="o", label=label, color=color, linewidth=2)
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(bands, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("変化率中央値 (%)")
ax.set_title("東京からの距離帯別 年次変化率")
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.2f}%"))

# 右: Donut Diff（コロナ禍累計 − コロナ前）
ax = axes[1]
diff = donut["donut_diff"]
colors_d = ["#4A90D9" if v >= 0 else "#E05C5C" for v in diff]
ax.bar(x, diff, color=colors_d, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(bands, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("コロナ禍累計 − コロナ前 (pp)")
ax.set_title("Donut Effect: コロナ禍の上乗せ変化率\n（青=コロナ前より上昇 / 赤=コロナ前より下落）")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.2f}"))

plt.suptitle("Donut Effect 検証（東京）", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "corona_donut_effect.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {OUT_DIR / 'corona_donut_effect.png'}")

## 5. 中間保存（集計CSVデータ）

In [ ]:
summary.to_csv(OUT_DIR / "corona_yearly_summary.csv")
dist_table.to_csv(OUT_DIR / "corona_district_table.csv")
donut.to_csv(OUT_DIR / "corona_donut_table.csv")

print("=== 保存ファイル ===")
for f in sorted(OUT_DIR.glob("corona_*")):
    print(f"  {f.name}")

In [ ]:
chg_cols = {
    "2018→2019": "chg_18_19",
    "2019→2020": "chg_19_20",
    "2020→2021": "chg_20_21",
    "2021→2022": "chg_21_22",
}

print(f"{'期間':<20} {'据え置き':>10} {'上昇':>10} {'下落':>10} {'合計':>8}")
print("-" * 62)
for label, col in chg_cols.items():
    s = panel[col].dropna()
    zero_pct = (s == 0).mean() * 100
    up_pct   = (s > 0).mean() * 100
    down_pct = (s < 0).mean() * 100
    total    = zero_pct + up_pct + down_pct
    print(f"{label:<20} {zero_pct:>9.1f}% {up_pct:>9.1f}% {down_pct:>9.1f}% {total:>7.1f}%")

## 6. 路線レベル 変動傾向マップ（4期間）

各路線の重心点を「上昇 / 据え置き / 下落」で色分けし、4期間を並べて比較する。

In [ ]:

# ── 路線レベル 変動傾向マップ（2×2 / 4期間）────────────────────────────
# 重心点に変換（LineString → Point）
panel_pt = panel.copy()
panel_pt["geometry"] = panel.geometry.centroid
panel_pt = panel_pt.to_crs(epsg=3857)

periods = {
    "① コロナ前\n2018→2019":    "chg_18_19",
    "② コロナ発生年\n2019→2020": "chg_19_20",
    "③ コロナ禍\n2020→2021":    "chg_20_21",
    "④ 回復期\n2021→2022":      "chg_21_22",
}

# サンプリング設定（下落・上昇は全件、据え置きは間引き）
SAMPLE_FLAT = 80_000   # 据え置きは間引いて背景として描画
SEED = 42

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flatten()

for ax, (label, col) in zip(axes, periods.items()):
    sub = panel_pt[[col, "geometry"]].dropna(subset=[col])

    up   = sub[sub[col] >  0]
    flat = sub[sub[col] == 0]
    dn   = sub[sub[col] <  0]

    # 据え置きをサンプリング（多すぎると塗りつぶされる）
    if len(flat) > SAMPLE_FLAT:
        flat = flat.sample(SAMPLE_FLAT, random_state=SEED)

    # 描画順: 据え置き（背景）→ 上昇 → 下落（前面）
    flat.plot(ax=ax, color="#CCCCCC", markersize=0.3, alpha=0.25)
    up.plot(  ax=ax, color="#D94A4A", markersize=0.5, alpha=0.45)
    dn.plot(  ax=ax, color="#3A7FC1", markersize=0.8, alpha=0.65)

    try:
        import contextily as ctx
        ctx.add_basemap(ax, crs=panel_pt.crs.to_string(),
                        source=ctx.providers.CartoDB.DarkMatter, zoom=5, alpha=0.45)
    except Exception:
        pass

    # 凡例
    import matplotlib.lines as mlines
    leg = [
        mlines.Line2D([], [], color="#D94A4A", marker="o", ls="none",
                      markersize=6, label=f"上昇  ({len(up):,}件)"),
        mlines.Line2D([], [], color="#CCCCCC", marker="o", ls="none",
                      markersize=6, label=f"据え置き ({len(sub[sub[col]==0]):,}件)"),
        mlines.Line2D([], [], color="#3A7FC1", marker="o", ls="none",
                      markersize=6, label=f"下落  ({len(dn):,}件)"),
    ]
    ax.legend(handles=leg, loc="lower left", fontsize=8,
              frameon=True, framealpha=0.8)

    ax.set_title(label, fontsize=13, fontweight="bold", pad=10, color="white"
                 if True else "black")
    ax.set_axis_off()

fig.patch.set_facecolor("#1a1a2e")
for ax in axes:
    ax.set_facecolor("#1a1a2e")
    ax.title.set_color("white")

plt.suptitle("路線レベル 変動傾向マップ（コロナ前後4期間）\n赤=上昇 / 灰=据え置き / 青=下落",
             fontsize=15, color="white", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "corona_map_road.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"保存: {OUT_DIR / 'corona_map_road.png'}")


## 7. 路線レベルの変動タイプ分類

各路線を「コロナ前後の変化率パターン」で4タイプに分類する。

| タイプ | 条件 | 意味 |
|--------|------|------|
| 構造的下落 | コロナ前（2018→2019）に下落 | コロナとは無関係の人口減少・地域衰退 |
| コロナ型下落 | コロナ前は安定、コロナ禍に下落、未回復 | コロナで傷つき回復していない |
| 回復型 | コロナ禍に下落、回復期に上昇 | コロナの影響を受けたが回復 |
| 安定・上昇 | コロナ禍も下落しなかった | コロナの影響を受けなかった |

In [ ]:

# ── 分類ロジック ─────────────────────────────────────────────────────────
def classify_road(row):
    pre   = row["chg_18_19"]   # コロナ前変化率
    covid = row["chg_20_21"]   # コロナ禍変化率
    rec   = row["chg_21_22"]   # 回復期変化率

    if pd.isna(pre) or pd.isna(covid) or pd.isna(rec):
        return "不明"
    if pre < 0:
        return "構造的下落"
    if covid < 0:
        if rec > 0:
            return "回復型"
        else:
            return "コロナ型下落"
    return "安定・上昇"

panel["road_type"] = panel.apply(classify_road, axis=1)

type_counts = panel["road_type"].value_counts()
total = type_counts.sum()
print("=== 路線タイプ別件数 ===")
for t, n in type_counts.items():
    print(f"  {t:<12} : {n:>9,}件  ({n/total*100:.1f}%)")


In [ ]:

# ── 都道府県別タイプ構成比 ────────────────────────────────────────────────
type_order = ["安定・上昇", "回復型", "コロナ型下落", "構造的下落"]

cross = (
    panel.groupby(["pref_code", "road_type"])
    .size()
    .unstack(fill_value=0)
)
for t in type_order:
    if t not in cross.columns:
        cross[t] = 0
cross = cross[type_order]
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
cross_pct_reset = cross_pct.reset_index()

# ── 差分列: コロナ型下落 − 構造的下落 ───────────────────────────────────
cross_pct_reset["diff"] = cross_pct_reset["コロナ型下落"] - cross_pct_reset["構造的下落"]

# 都道府県境界と結合
japan_url = "https://raw.githubusercontent.com/dataofjapan/land/master/japan.geojson"
import geopandas as gpd
japan = gpd.read_file(japan_url)
japan["pref_code"] = japan["id"].astype(int)

gdf_map  = japan.merge(cross_pct_reset, on="pref_code", how="left")
gdf_3857 = gdf_map.to_crs(epsg=3857)

# ── 描画 ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))

abs_max = cross_pct_reset["diff"].abs().max()
norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)

gdf_3857.plot(
    column="diff", ax=ax,
    cmap="RdBu_r", norm=norm,
    edgecolor="white", linewidth=0.5,
    missing_kwds={"color": "lightgrey"},
)

try:
    import contextily as ctx
    ctx.add_basemap(ax, crs=gdf_3857.crs.to_string(),
                    source=ctx.providers.CartoDB.Positron, zoom=5, alpha=0.35)
except Exception:
    pass

sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, shrink=0.7)
cb.set_label("コロナ型下落 − 構造的下落 (%pt)", fontsize=10)
cb.set_ticks([-abs_max, 0, abs_max])
cb.set_ticklabels(
    [f"構造的下落が多い\n−{abs_max:.0f}pt", "均衡", f"コロナ型下落が多い\n+{abs_max:.0f}pt"],
    fontsize=9
)

ax.set_axis_off()
ax.set_title(
    "下落の「原因」の地域差\n"
    "赤 = コロナ起因の下落が多い都道府県\n"
    "青 = コロナ前からの構造的下落が多い都道府県",
    fontsize=13, fontweight="bold", pad=12
)

plt.tight_layout()
plt.savefig(OUT_DIR / "corona_road_type_map.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {OUT_DIR / 'corona_road_type_map.png'}")


## 8. コロナと無関係に「ずっと上昇」「ずっと下落」だった路線

全4期間（2018→2019, 2019→2020, 2020→2021, 2021→2022）を通じて
一貫して上昇または下落し続けた路線を抽出し、地域パターンを確認する。

In [ ]:

chg_cols = ["chg_18_19", "chg_19_20", "chg_20_21", "chg_21_22"]

# 全4期間で変化率が揃っている路線のみ対象
mask_valid = panel[chg_cols].notna().all(axis=1)
p = panel[mask_valid].copy()

# ずっと上昇: 全4期間 > 0
always_up   = p[(p[chg_cols] > 0).all(axis=1)]
# ずっと下落: 全4期間 < 0
always_down = p[(p[chg_cols] < 0).all(axis=1)]

total = len(p)
print(f"対象路線総数      : {total:>10,}")
print(f"ずっと上昇（全4期）: {len(always_up):>10,}件  ({len(always_up)/total*100:.1f}%)")
print(f"ずっと下落（全4期）: {len(always_down):>10,}件  ({len(always_down)/total*100:.1f}%)")

# 都道府県別集計
def pref_summary(gdf, label):
    g = gdf.groupby("pref_code").size().reset_index(name=label)
    tot = p.groupby("pref_code").size().reset_index(name="total")
    m = tot.merge(g, on="pref_code", how="left").fillna(0)
    m[f"{label}_pct"] = m[label] / m["total"] * 100
    return m.sort_values(f"{label}_pct", ascending=False)

up_pref   = pref_summary(always_up,   "always_up")
down_pref = pref_summary(always_down, "always_down")

print("\n=== ずっと上昇が多い都道府県 TOP10 ===")
print(up_pref[["pref_code","always_up","total","always_up_pct"]].head(10).to_string(index=False))

print("\n=== ずっと下落が多い都道府県 TOP10 ===")
print(down_pref[["pref_code","always_down","total","always_down_pct"]].head(10).to_string(index=False))


In [ ]:

# ── マップ描画（ずっと上昇 / ずっと下落 / その他）────────────────────────
import geopandas as gpd

# 重心点に変換
def to_pt(gdf):
    g = gdf.copy()
    g["geometry"] = gdf.geometry.centroid
    return g.to_crs(epsg=3857)

pt_up   = to_pt(always_up)
pt_down = to_pt(always_down)

# 背景用: それ以外の路線（サンプリング）
other_idx = p.index.difference(always_up.index).difference(always_down.index)
other_sample = to_pt(p.loc[other_idx].sample(min(100_000, len(other_idx)), random_state=42))

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#1a1a2e")

# 描画順: 背景 → ずっと下落 → ずっと上昇
other_sample.plot(ax=ax, color="#444466", markersize=0.2, alpha=0.2)
pt_down.plot(    ax=ax, color="#3A7FC1", markersize=0.005, alpha=1)
pt_up.plot(      ax=ax, color="#E8534A", markersize=0.005, alpha=1)

try:
    import contextily as ctx
    ctx.add_basemap(ax, crs=pt_up.crs.to_string(),
                    source=ctx.providers.CartoDB.DarkMatter, zoom=5, alpha=0.4)
except Exception:
    pass

import matplotlib.lines as mlines
leg = [
    mlines.Line2D([], [], color="#E8534A", marker="o", ls="none",
                  markersize=7, label=f"ずっと上昇（全4期間）  {len(pt_up):,}件"),
    mlines.Line2D([], [], color="#3A7FC1", marker="o", ls="none",
                  markersize=7, label=f"ずっと下落（全4期間）  {len(pt_down):,}件"),
    mlines.Line2D([], [], color="#444466", marker="o", ls="none",
                  markersize=5, label="その他"),
]
ax.legend(handles=leg, loc="lower left", fontsize=10,
          frameon=True, framealpha=0.85, facecolor="#222233", labelcolor="white")

ax.set_axis_off()
ax.set_title("コロナと無関係に「ずっと上昇」「ずっと下落」だった路線（2018〜2022 全4期間）",
             fontsize=13, fontweight="bold", color="white", pad=12)

plt.tight_layout()
plt.savefig(OUT_DIR / "corona_always_trend.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"保存: {OUT_DIR / 'corona_always_trend.png'}")


## 9. 全出力ファイル一覧

In [ ]:

# CSV保存（section 5 の中間保存に追加して最終版を保存）
summary.to_csv(OUT_DIR / "corona_yearly_summary.csv")
dist_table.to_csv(OUT_DIR / "corona_district_table.csv")
donut.to_csv(OUT_DIR / "corona_donut_table.csv")

print("=== 全出力ファイル ===")
categories = {
    "集計CSV": list(OUT_DIR.glob("corona_*.csv")),
    "グラフPNG": list(OUT_DIR.glob("corona_*.png")),
}
for cat, files in categories.items():
    print(f"\n【{cat}】")
    for f in sorted(files):
        print(f"  {f.name}")
